# 🃏 Poker AI v4 — Kaggle Trainer

| GPU | Ingyenes idő | Mentés |
|---|---|---|
| T4 × 2 vagy P100 | **30 óra / hét** | `/kaggle/working/` → Output letöltés |

### Elindítás előtt:
- **Accelerator**: *Session Options → Accelerator → GPU T4 × 2*
- **Internet**: *Session Options → Internet → On*

### Gyors útmutató:
1. **Lépés 1** — GPU-ellenőrzés + dependenciák
2. **Lépés 2** — Kódbázis betöltése (git clone vagy dataset)
3. **Lépés 3** — (Opcionális) WandB / Kaggle Secrets
4. **Lépés 4** — ⚙️ Konfiguráció — **ITT ÁLLÍTS!**
5. **Lépés 5** — 🚀 Tréning indítása
6. **Lépés 6** — 💾 Checkpoint mentése ZIP-be

---
## Lépés 1 — GPU-ellenőrzés és dependenciák

In [ ]:
import torch

# ── GPU-ellenőrzés ────────────────────────────────────────────────────────
print('=' * 52)
print('  GPU ÁLLAPOT')
print('=' * 52)
print(f'  PyTorch:  {torch.__version__}')
print(f'  CUDA:     {torch.cuda.is_available()}')

if torch.cuda.is_available():
    for i in range(torch.cuda.device_count()):
        props = torch.cuda.get_device_properties(i)
        vram = props.total_memory / 1e9
        print(f'  GPU {i}: {props.name}  ({vram:.1f} GB VRAM)')
    print()
    # NUM_ENVS javaslat
    total_vram = sum(
        torch.cuda.get_device_properties(i).total_memory
        for i in range(torch.cuda.device_count())
    ) / 1e9
    suggested_envs = 512 if total_vram >= 14 else 256
    print(f'  ℹ️  Javasolt NUM_ENVS: {suggested_envs}  (összes VRAM: {total_vram:.1f} GB)')
else:
    print()
    print('  ⚠️  FIGYELEM: Nincs GPU! Menj: Session Options → Accelerator → GPU T4 × 2')

print('=' * 52)

# ── Dependenciák telepítése ───────────────────────────────────────────────
# A requirements.txt alapján telepítünk (a kódbázis betöltése után).
# Ha a kódbázis még nincs meg, csak az alapokat telepítjük most:
import subprocess
subprocess.run(
    'pip install -q rlcard>=1.0.5 tensorboard>=2.13.0',
    shell=True, check=True
)
print('\n✅ Alap dependenciák telepítve.')

---
## Lépés 2 — Kódbázis betöltése

**Két módszer — válaszd a neked megfelelőt:**

### A) Git clone (ajánlott — mindig a legfrissebb kód)
Beállítsd a `REPO_URL` változót a cellában, és add meg a tokened ha privát a repo.

### B) Kaggle Dataset (offline, ha nincs internet-engedély)
1. Tömörítsd a projektet: `zip -r poker_ai_v4.zip . --exclude='*.pth' --exclude='__pycache__/*'`
2. Töltsd fel: [kaggle.com/datasets/new](https://www.kaggle.com/datasets/new) → Név: `poker-ai-v4-code` → **Private**
3. Ebben a notebookban: **+ Add Data → Your Datasets → poker-ai-v4-code**
4. Állítsd `USE_GIT_CLONE = False`-ra lent

In [ ]:
import os, shutil, zipfile, glob, subprocess

# ════════════════════════════════════════════════
USE_GIT_CLONE  = True                             # True = git clone, False = dataset

# --- A) Git clone beállítások ---
REPO_URL       = 'https://github.com/Mikidikilang/PokerAI.git'
# Privát repo esetén: 'https://<TOKEN>@github.com/user/repo.git'
REPO_BRANCH    = 'main'

# --- B) Kaggle Dataset beállítások ---
DATASET_NAME   = 'poker-ai-v4-code'               # Kaggle dataset neve
# ════════════════════════════════════════════════

CODE_DIR = '/kaggle/working/poker_ai'

# ── Már létező kód ellenőrzése ────────────────────────────────────────────
if os.path.exists(os.path.join(CODE_DIR, 'train.py')):
    print(f'✅ Kód már megvan: {CODE_DIR}')

elif USE_GIT_CLONE:
    # ── A) Git clone ─────────────────────────────────────────────────────
    if os.path.exists(CODE_DIR):
        shutil.rmtree(CODE_DIR)
    result = subprocess.run(
        f'git clone --branch {REPO_BRANCH} --depth 1 {REPO_URL} {CODE_DIR}',
        shell=True, capture_output=True, text=True
    )
    if result.returncode != 0:
        print('❌ Git clone hiba:')
        print(result.stderr)
        raise RuntimeError('Git clone sikertelen! Ellenőrizd a REPO_URL-t és az internet-hozzáférést.')
    print(f'✅ Git clone kész: {CODE_DIR}')

else:
    # ── B) Kaggle Dataset kibontása ───────────────────────────────────────
    dataset_path = f'/kaggle/input/{DATASET_NAME}'
    if not os.path.exists(dataset_path):
        available = os.listdir('/kaggle/input') if os.path.exists('/kaggle/input') else []
        raise FileNotFoundError(
            f'Dataset nem található: {dataset_path}\n'
            f'Elérhető datasetek: {available}\n\n'
            f'→ Kattints a + Add Data gombra és add hozzá: {DATASET_NAME}'
        )
    zips = glob.glob(os.path.join(dataset_path, '*.zip'))
    if not zips:
        raise FileNotFoundError(
            f'Nincs .zip a datasetben! Tartalom: {os.listdir(dataset_path)}'
        )
    tmp = '/kaggle/working/_tmp_extract'
    if os.path.exists(tmp): shutil.rmtree(tmp)
    with zipfile.ZipFile(zips[0], 'r') as z:
        z.extractall(tmp)
    found = None
    for root, dirs, files in os.walk(tmp):
        if 'train.py' in files:
            found = root
            break
    if found is None:
        raise RuntimeError('train.py nem található a zip-ben!')
    if os.path.exists(CODE_DIR): shutil.rmtree(CODE_DIR)
    shutil.copytree(found, CODE_DIR)
    shutil.rmtree(tmp)
    print(f'✅ Kód kibontva: {CODE_DIR}')

# ── requirements.txt telepítése ───────────────────────────────────────────
req_path = os.path.join(CODE_DIR, 'requirements.txt')
if os.path.exists(req_path):
    subprocess.run(f'pip install -q -r {req_path}', shell=True, check=True)
    print('✅ requirements.txt dependenciák telepítve.')

print(f'\nFőbb fájlok: {[f for f in os.listdir(CODE_DIR) if not f.startswith(".")][:10]}')

---
## Lépés 3 — (Opcionális) WandB / Kaggle Secrets

Ha **Weights & Biases** logolást szeretnél, add hozzá az API kulcsot Kaggle Secrets-ben:
- Kaggle → Account → Secrets → **+ Add New Secret**
- Kulcs neve: `WANDB_API_KEY`
- Majd ebben a notebookban engedélyezd: Notebook Options → Secrets → `WANDB_API_KEY` ✓

Ha nem használsz WandB-t, ezt a cellát átugorhatod.

In [ ]:
import os

USE_WANDB = False  # ← Állítsd True-ra ha WandB-t akarsz használni

if USE_WANDB:
    try:
        from kaggle_secrets import UserSecretsClient
        secrets = UserSecretsClient()
        wandb_key = secrets.get_secret('WANDB_API_KEY')
        os.environ['WANDB_API_KEY'] = wandb_key
        print('✅ WANDB_API_KEY sikeresen betöltve a Kaggle Secrets-ből.')

        import subprocess
        subprocess.run('pip install -q wandb', shell=True, check=True)
        import wandb
        wandb.login(key=wandb_key)
        print(f'✅ WandB bejelentkezve: {wandb.api.viewer()["entity"]}')

    except Exception as e:
        print(f'⚠️  WandB bejelentkezés sikertelen: {e}')
        print('   → Ellenőrizd, hogy a Secrets-ben van-e WANDB_API_KEY és engedélyezted-e.')
        USE_WANDB = False
else:
    print('ℹ️  WandB kikapcsolva. (USE_WANDB = False)')

---
## Lépés 4 — ⚙️ Konfiguráció

**ITT ÁLLÍTS!** — A cella végén a `ModelManager` kiírja a `config.json`-t, amit a `train.py` közvetlenül beolvas. Így minden paraméter — köztük a `num_envs` és `learning_rate` is — biztosan érvényesül.

| Paraméter | Leírás | Javasolt érték |
|---|---|---|
| `MODEL_NAME` | Modell azonosítója | `'2max'`, `'6max'`, stb. |
| `NUM_PLAYERS` | Játékosok száma | `2`, `6`, `9` |
| `TRAINING_MODE` | Tréning stílus | `'exploitative'`, `'selfplay'`, `'aggressive'` |
| `EPISODES` | Epizódok ebben a sessionben | `1_000_000` – `2_000_000` |
| `SAVE_FREQ` | Mérföldkő intervallum | `500_000` (Kaggle-on) |
| `TEST_HANDS` | Mérföldkő-teszt kézszám | `500` (Kaggle-on — ne adj meg sokat!) |
| `NUM_ENVS` | Párhuzamos env-ek | `256` (T4), `512` (P100) |
| `LEARNING_RATE` | Adam optimizer LR | `3e-4` (default) |
| `HIDDEN_SIZE` | Modell szélessége | `512` (default) |

In [ ]:
import os, sys, glob, zipfile, shutil

# ════════════════════════════════════════════════════════════
MODEL_NAME      = '2max'           # Modell neve
NUM_PLAYERS     = 2                # Játékosok száma: 2 / 6 / 9
TRAINING_MODE   = 'exploitative'   # 'exploitative' | 'selfplay' | 'aggressive'
EPISODES        = 1_000_000        # Epizódok ebben a sessionben
SAVE_FREQ       = 500_000          # Mérföldkő intervallum (Kaggle-on: 500_000)
TEST_HANDS      = 500              # Teszt kézszám mérföldkőnél ⚠️ tartsd alacsonyan!
NUM_ENVS        = 256              # T4: 256 | P100: 512
LEARNING_RATE   = 3e-4             # Adam LR
HIDDEN_SIZE     = 512              # Modell rejtett réteg mérete
# ════════════════════════════════════════════════════════════

# Előző session checkpointjának visszatöltése (opcionális)
PREV_CHECKPOINT_DATASET = ''       # pl. 'pokerai-2max-v1' — hagyd üresen ha nincs

# ── Output mappák ─────────────────────────────────────────────────────────
OUTPUT_BASE = '/kaggle/working/PokerAI_Models'
MODEL_DIR   = os.path.join(OUTPUT_BASE, 'models', MODEL_NAME)

# ── sys.path beállítása ───────────────────────────────────────────────────
if CODE_DIR not in sys.path:
    sys.path.insert(0, CODE_DIR)

# ── ModelManager: config.json előre megírása ─────────────────────────────
# A train.py ezt olvassa be — így num_envs, learning_rate stb. biztosan
# érvényesülnek, nem a CONFIG_DEFAULTS veszi át az irányítást.
# Megjegyzés: a TRAINING_MODE preset (exploitative/selfplay/aggressive)
# csak az entropy és bot_pool értékeket írja felül — num_envs és
# learning_rate érintetlenül marad.
from training.model_manager import ModelManager

mgr = ModelManager(OUTPUT_BASE)
mgr.ensure_model_dir(MODEL_NAME, NUM_PLAYERS)

cfg_data = mgr.load_config(MODEL_NAME)
cfg_data['num_players'] = NUM_PLAYERS
cfg_data['config'].update({
    'num_envs':           NUM_ENVS,
    'learning_rate':      LEARNING_RATE,
    'hidden_size':        HIDDEN_SIZE,
    'milestone_interval': SAVE_FREQ,
    'milestone_hands':    TEST_HANDS,
    'training_style':     TRAINING_MODE,
})
mgr.save_config(MODEL_NAME, cfg_data)
print(f'✅ config.json megírva: {mgr.config_path(MODEL_NAME)}')

# PTH elérési út a ModelManager-től (automatikusan kezeli a névkonvenciókat)
PTH_PATH = mgr.pth_path(MODEL_NAME)

# ── Előző checkpoint visszatöltése ────────────────────────────────────────
if PREV_CHECKPOINT_DATASET and not os.path.exists(PTH_PATH):
    ds = f'/kaggle/input/{PREV_CHECKPOINT_DATASET}'
    if os.path.exists(ds):
        zips = glob.glob(os.path.join(ds, '*.zip'))
        pths = glob.glob(os.path.join(ds, '**', '*.pth'), recursive=True)
        if zips:
            with zipfile.ZipFile(zips[0], 'r') as z:
                z.extractall('/kaggle/working/')
            print(f'✅ Előző checkpoint zip visszatöltve: {zips[0]}')
        elif pths:
            shutil.copy2(pths[0], PTH_PATH)
            print(f'✅ Előző checkpoint visszatöltve: {PTH_PATH}')
        else:
            print(f'⚠️  Nem találtam .pth fájlt a datasetben: {ds}')
    else:
        print(f'⚠️  Dataset nem elérhető: {ds}')

# ── Összefoglaló ──────────────────────────────────────────────────────────
print('═' * 56)
print('  KONFIGURÁCIÓ ÖSSZEFOGLALÓ')
print('═' * 56)
print(f'  Modell:         {MODEL_NAME}  ({NUM_PLAYERS} játékos)')
print(f'  Tréning mód:    {TRAINING_MODE}')
print(f'  Epizódok:       {EPISODES:,}')
print(f'  Mérföldkő:      minden {SAVE_FREQ:,} epizódonként')
print(f'  Teszt kéz:      {TEST_HANDS}  ⚠️  tartsd alacsonyan!')
print(f'  GPU envs:       {NUM_ENVS}')
print(f'  Tanulási ráta:  {LEARNING_RATE}')
print(f'  Hidden size:    {HIDDEN_SIZE}')
print('─' * 56)
print(f'  Output:   {OUTPUT_BASE}')
print(f'  PTH:      {PTH_PATH}')
print('═' * 56)

if os.path.exists(PTH_PATH):
    from utils.checkpoint_utils import safe_load_checkpoint
    ck = safe_load_checkpoint(PTH_PATH, map_location='cpu')
    if isinstance(ck, dict) and 'episodes_trained' in ck:
        print(f'  ♻️  Folytatás: {ck["episodes_trained"]:,} epizódtól')
    else:
        print(f'  ♻️  Meglévő checkpoint betöltve.')
else:
    print(f'  🆕 Új tréning indul.')

---
## Lépés 5 — 🚀 Tréning indítása

A tréninget a `train.py` headless CLI kezeli — pontosan ugyanúgy működik mint helyi RunPod futtatáskor.
A checkpoint és a napló automatikusan a `/kaggle/working/PokerAI_Models/` mappába kerül.

> **Megszakítás esetén (Ctrl+C / timeout):** a legutóbbi mérföldkő checkpoint megmarad — futtasd a 6. lépést a mentéshez!

In [ ]:
import subprocess, sys, os

# ── train.py CLI argumentumok összeállítása ───────────────────────────────
cmd = [
    sys.executable,
    os.path.join(CODE_DIR, 'train.py'),
    '--model_name',  MODEL_NAME,
    '--players',     str(NUM_PLAYERS),
    '--mode',        TRAINING_MODE,
    '--iterations',  str(EPISODES),
    '--save_freq',   str(SAVE_FREQ),
    '--test_hands',  str(TEST_HANDS),
    '--output_dir',  OUTPUT_BASE,
]

# Opcionális: learning_rate és num_envs átadása config-json-on keresztül
# (a train.py ezeket a model config-ból veszi, de felülírhatók a config.json-ban
#  amit a 4. lépés beállított — nincs szükség extra argumentumra)

print('Indítás...')
print('Parancs: ' + ' '.join(str(a) for a in cmd))
print('=' * 56)

env = {
    **os.environ,
    'PYTHONPATH':      CODE_DIR,
    'PYTHONUNBUFFERED': '1',
}

process = subprocess.Popen(
    cmd,
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
    bufsize=1,
    env=env,
)

try:
    for line in process.stdout:
        print(line, end='', flush=True)
    process.wait()
except KeyboardInterrupt:
    print('\n⚠️  Megszakítva (Ctrl+C) — a checkpoint megmarad.')
    print('   → Futtasd a 6. lépést a mentéshez!')
    process.terminate()
    process.wait()

rc = process.returncode
if rc == 0:
    print('\n✅ Tréning sikeresen befejezve!')
    print('→ Futtasd a 6. lépést a checkpoint mentéséhez!')
elif rc is None:
    print('\n⚠️  Folyamat megszakítva — checkpoint megmaradt.')
    print('→ Futtasd a 6. lépést a mentéshez!')
else:
    print(f'\n❌ Hiba! Visszatérési kód: {rc}')
    print('   Ellenőrizd a fenti hibaüzenetet.')

---
## Lépés 6 — 💾 Checkpoint becsomagolása és mentése

**Ezt minden session végén futtasd!**

Utána kattints a **Save Version** (Commit) gombra — a `/kaggle/working/` tartalmát az Output szekcióból le tudod tölteni.

Ha folytatni akarod a következő sessionben:
1. Töltsd le a ZIP-et
2. Töltsd fel új Kaggle Datasetként
3. Add hozzá a notebookhoz (+ Add Data)
4. Írd be a dataset nevét a `PREV_CHECKPOINT_DATASET` változóba (4. lépés)

In [ ]:
import os, shutil, json, glob, sys

if CODE_DIR not in sys.path:
    sys.path.insert(0, CODE_DIR)

# ── Checkpoint állapot összefoglaló ───────────────────────────────────────
print('═' * 56)
print('  CHECKPOINT ÁLLAPOT')
print('═' * 56)

if os.path.exists(PTH_PATH):
    from utils.checkpoint_utils import safe_load_checkpoint
    ck   = safe_load_checkpoint(PTH_PATH, map_location='cpu')
    ep   = ck.get('episodes_trained', 0) if isinstance(ck, dict) else 0
    t    = ck.get('time_spent', 0)       if isinstance(ck, dict) else 0
    size = os.path.getsize(PTH_PATH) / 1e6
    print(f'  ✅ {os.path.basename(PTH_PATH)}')
    print(f'     Epizódok:    {ep:,}')
    print(f'     Tréning idő: {t/3600:.1f} óra')
    print(f'     Fájlméret:   {size:.1f} MB')
else:
    print(f'  ❌ Nincs checkpoint: {PTH_PATH}')

# ── Mérföldkövek listája ──────────────────────────────────────────────────
tests_dir = os.path.join(MODEL_DIR, 'tests')
ms_dirs   = sorted(glob.glob(os.path.join(tests_dir, '*/')), reverse=True)
print(f'\n  Mérföldkövek: {len(ms_dirs)} db')
for md in ms_dirs[:5]:
    name  = os.path.basename(md.rstrip('/'))
    grade = '?'
    for jf in glob.glob(os.path.join(md, '*.json')):
        try:
            with open(jf) as f:
                d = json.load(f)
            grade = d.get('summary', {}).get('grade', '?')
        except Exception:
            pass
    pth_ok = '✅' if glob.glob(os.path.join(md, '*.pth')) else '❌'
    print(f'    {pth_ok} {name:<35}  grade: {grade}')

# ── ZIP elkészítése ───────────────────────────────────────────────────────
zip_base = f'/kaggle/working/pokerai_{MODEL_NAME}_checkpoint'
shutil.make_archive(zip_base, 'zip', OUTPUT_BASE)
zip_size = os.path.getsize(zip_base + '.zip') / 1e6

print(f'\n  ✅ ZIP elkészítve!')
print(f'     Elérési út: {zip_base}.zip')
print(f'     Méret:      {zip_size:.1f} MB')

print()
print('═' * 56)
print('  KÖVETKEZŐ LÉPÉSEK:')
print('═' * 56)
print('  1.  Kattints: Save Version  (jobb felső sarok)')
print('  2.  Mentés után: Notebook oldal → Output')
print(f'  3.  Töltsd le: pokerai_{MODEL_NAME}_checkpoint.zip')
print()
print('  Ha FOLYTATNI akarod a következő sessionben:')
print('  4.  Töltsd fel a ZIP-et új Kaggle Datasetként')
print('  5.  Add hozzá a notebookhoz (+ Add Data)')
print('  6.  Írd be a dataset nevét PREV_CHECKPOINT_DATASET-be (4. lépés)')